# 01 — Limpeza dos dados

**Objetivo:** transformar a base original (`data/raw/ICB_DADOS_2026.xlsx`) numa base limpa e padronizada, usada por todas as etapas seguintes.

A lógica fica em `src/cleaning.py`; este notebook só chama as funções e mostra o que cada etapa fez. Decisões aplicadas: **2** (produtos e nomes) e **10** (critérios de limpeza), em `docs/decisoes.md`.

Saídas:
- `data/processed/icb_limpo.csv` — base limpa (lida pelas etapas 02, 04 e 05 com `data_loading.carregar_base_limpa`);
- `reports/tables/limpeza_resumo.csv` — linhas removidas em cada etapa (para o relatório).

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote src/ a partir da pasta notebooks/
sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src import cleaning, config, data_loading

## 1. Carregar a base original

Lê a planilha sem nenhuma alteração. São cerca de 46 mil coletas de preço: uma linha por produto, marca e mercado em cada visita.

In [2]:
bruta = data_loading.carregar_icb()
print(f"{len(bruta):,} linhas, {bruta.shape[1]} colunas")
bruta.head()

46,264 linhas, 8 colunas


,Mes_Ano_Data,Data_Coleta,Estab,Produto,Marca,Preco,Quantidade,PPK
0,2022-09-01,2022-09-23,SUP_33,Carne Bovina Coxão Mole,NaN,54.20,1.0,54.20
1,2022-09-01,2022-09-12,SUP_15,Carne Bovina Coxão Mole,NaN,53.90,1.0,53.90
2,2022-09-01,2022-09-21,SUP_09,Carne Bovina Coxão Mole,NaN,53.90,1.0,53.90
3,2022-09-01,2022-09-23,SUP_12,Carne Bovina Coxão Mole,NaN,51.99,1.0,51.99
4,2022-09-01,2022-09-12,SUP_20,Carne Bovina Coxão Mole,NaN,51.90,1.0,51.90


## 2. Diagnóstico

Antes de limpar, olhamos que problemas existem. Cada achado abaixo justifica uma etapa da limpeza.

**Valores faltantes.** `Data_Coleta` está vazia a partir de nov/2025 (a coluna deixou de ser preenchida); não é problema, porque a análise usa o mês (`Mes_Ano_Data`). `Marca` falta em hortifrúti, carne e pão, que não têm marca.

In [3]:
bruta.isna().sum()

Mes_Ano_Data       0
Data_Coleta     9818
Estab              0
Produto            0
Marca           9346
Preco              0
Quantidade         0
PPK                0
dtype: int64

In [4]:
# Proporção de Data_Coleta vazia por mês (só os meses com falta)
falta_coleta = bruta.groupby(bruta[config.COL_MES].dt.to_period("M"))[config.COL_DATA_COLETA].apply(lambda s: s.isna().mean())
falta_coleta[falta_coleta > 0]

Mes_Ano_Data
2025-11    1.0
2025-12    1.0
2026-01    1.0
2026-02    1.0
2026-03    1.0
2026-04    1.0
2026-05    1.0
2026-06    1.0
2026-07    1.0
2026-08    1.0
Freq: M, Name: Data_Coleta, dtype: float64

**Nomes de produto duplicados.** O mesmo produto aparece com nomes diferentes ("Pão" e "Pão Francês"). O mapa de unificação está em `config.MAPA_NOMES_PRODUTOS`.

In [5]:
bruta[config.COL_PRODUTO].value_counts()

Produto
Café                       5378
Arroz                      5305
Manteiga                   4875
Feijão                     4790
Leite                      4651
Farinha de Trigo           4196
Óleo                       3165
Açúcar                     3101
Tomate                     1332
Batata                     1325
Banana Prata               1225
Banana Nanica              1221
Carne Bovina Coxão Mole    1216
Pão Francês                1054
Macarrão com Ovos          1053
Ovo                         462
Frango Peito                443
Frango Sobrecoxa            435
Carne Bovina Acem           407
Carne Suína Pernil          295
Pernil                      121
Farinha                      66
Macarrão                     52
Carne Pernil                 23
Carne Coxão Mole             21
Carne Acem                   19
Pão                          19
Carne Bovina Patinho          7
Carne Bovina Músculo          5
Carne Bovina Alcatra          2
Name: count, dtype: int64

**Preços impossíveis.** O PPK (preço por kg/unidade) vai de R$ 0,10 a R$ 999. Esses extremos são erros de digitação, não preços reais.

In [6]:
bruta.sort_values(config.COL_PPK).iloc[[0, 1, 2, -2, -1]]

,Mes_Ano_Data,Data_Coleta,Estab,Produto,Marca,Preco,Quantidade,PPK
35918,2025-10-01,2025-10-17,SUP_47,Pão Francês,NaN,0.1,1.0,0.1
38570,2025-12-01,NaT,SUP_17,Frango Sobrecoxa,NaN,0.1,1.0,0.1
37310,2025-11-01,NaT,SUP_17,Carne Bovina Coxão Mole,NaN,0.1,1.0,0.1
2657,2022-12-01,2022-12-16,SUP_01,Manteiga,Teixeira,29.5,0.2,147.5
22452,2024-10-01,2024-10-07,SUP_32,Banana Nanica,NaN,999.0,1.0,999.0


**Embalagens fora do padrão.** Quase todo produto tem uma embalagem dominante. Na manteiga, 97% é pote de 200 g; os de 250/500 g têm PPK menor e muitos são margarina (Qualy, Claybom, Doriana). Também há quantidades digitadas erradas (tomate com quantidade 20).

In [7]:
base_nomes = cleaning.padronizar_nomes_produtos(bruta)
base_nomes = cleaning.filtrar_produtos(base_nomes)
(base_nomes.groupby(config.COL_PRODUTO)[config.COL_QUANTIDADE]
    .value_counts(normalize=True).rename("proporcao").round(3)
    .reset_index().query("proporcao < 0.99 or proporcao == 1").head(30))

,Produto,Quantidade,proporcao
1,Arroz,2.00,0.002
2,Arroz,1.00,0.001
4,Açúcar,2.00,0.001
5,Açúcar,0.50,0.000
6,Açúcar,3.00,0.000
8,Banana Nanica,0.70,0.001
9,Banana Prata,1.00,1.000
10,Batata,1.00,1.000
12,Café,1.00,0.001
13,Café,0.25,0.000


**Linhas duplicadas.** Linhas idênticas em tudo (mercado, dia, produto, marca, preço, quantidade). Fariam o mesmo preço contar duas vezes na mediana do mercado.

In [8]:
print(f"linhas idênticas a outra: {bruta.duplicated().sum():,}")

linhas idênticas a outra: 1,743


## 3. Limpeza

`cleaning.limpar_base` executa as etapas nesta ordem (a ordem importa):

| Etapa | O que faz | Por quê |
|---|---|---|
| `converter_tipos` | datas como data, preços como número, sem espaços extras | base consistente |
| `padronizar_nomes_produtos` | "Pão" → "Pão Francês" etc. | decisão 2; antes do filtro, senão "Pão" seria perdido |
| `filtrar_produtos` | mantém os 14 produtos | decisão 2 |
| `remover_registros_invalidos` | sem mês/mercado/produto/PPK, ou valores ≤ 0 | não podem ser usados |
| `remover_duplicatas` | linhas idênticas viram uma | decisão 10: a mesma coleta não conta duas vezes |
| `filtrar_embalagem_padrao` | só a embalagem padrão de cada produto (`config.EMBALAGEM_PADRAO`) | decisão 10: comparar sempre o mesmo item |
| `remover_precos_atipicos` | PPK mais de 3× acima ou abaixo da mediana do produto no mês | decisão 10: remove erros de digitação (e ~10 possíveis promoções de hortifrúti, ver abaixo) |

In [9]:
limpa, resumo = cleaning.limpar_base(bruta)
resumo

,etapa,linhas,removidas
0,base original,46264,0
1,converter_tipos,46264,0
2,padronizar_nomes_produtos,46264,0
3,filtrar_produtos,42940,3324
4,remover_registros_invalidos,42940,0
5,remover_duplicatas,41258,1682
6,filtrar_embalagem_padrao,41063,195
7,remover_precos_atipicos,41029,34


### Quais preços atípicos saíram

Abaixo, os registros removidos na última etapa, com a razão entre o PPK e a mediana do produto no mês. A maioria é erro claro (R$ 0,10, R$ 999, arroz de 5 kg com preço de 1 kg, manteiga a R$ 3–4). Perto da razão 0,30–0,33 há também banana, batata e tomate baratos que podem ser promoção real; eles estão na mesma faixa de erros de arroz e manteiga, então não há limite único que separe os dois grupos (limitação registrada na decisão 10).

In [10]:
antes_atipicos = cleaning.filtrar_embalagem_padrao(
    cleaning.remover_duplicatas(
        cleaning.remover_registros_invalidos(
            cleaning.filtrar_produtos(
                cleaning.padronizar_nomes_produtos(cleaning.converter_tipos(bruta))))))

mediana_mes = antes_atipicos.groupby([config.COL_PRODUTO, config.COL_MES])[config.COL_PPK].transform("median")
removidos = antes_atipicos.assign(mediana_mes=mediana_mes, razao=antes_atipicos[config.COL_PPK] / mediana_mes)
removidos = removidos.drop(index=removidos.index.intersection(cleaning.remover_precos_atipicos(antes_atipicos).index))
removidos.sort_values("razao")[[config.COL_MES, config.COL_MERCADO, config.COL_PRODUTO, config.COL_MARCA,
                                config.COL_PRECO, config.COL_QUANTIDADE, config.COL_PPK, "mediana_mes", "razao"]]

,Mes_Ano_Data,Estab,Produto,Marca,Preco,Quantidade,PPK,mediana_mes,razao
37310,2025-11-01,SUP_17,Carne Bovina Coxão Mole,NaN,0.10,1.0,0.100000,49.440000,0.002023
35918,2025-10-01,SUP_47,Pão Francês,NaN,0.10,1.0,0.100000,16.990000,0.005886
36153,2025-10-01,SUP_36,Pão Francês,NaN,0.10,1.0,0.100000,16.990000,0.005886
2632,2022-11-01,SUP_29,Feijão,Solito,1.19,1.0,1.190000,8.980000,0.132517
7717,2023-05-01,SUP_33,Arroz,Qualitá,5.59,5.0,1.118000,5.118000,0.218445
29070,2025-03-01,SUP_04,Pão Francês,NaN,3.90,1.0,3.900000,16.990000,0.229547
7643,2023-05-01,SUP_33,Arroz,Camil,5.99,5.0,1.198000,5.118000,0.234076
42362,2026-05-01,SUP_11,Arroz,Broto Legal,5.98,5.0,1.196000,5.044000,0.237113
34399,2025-08-01,SUP_06,Manteiga,Outro,3.40,0.2,17.000000,70.725000,0.240368
947,2022-09-01,SUP_06,Manteiga,Claybom,3.15,0.2,15.750000,63.200000,0.249209


### Linhas por produto antes e depois

In [11]:
pd.DataFrame({
    "antes": base_nomes[config.COL_PRODUTO].value_counts(),
    "depois": limpa[config.COL_PRODUTO].value_counts(),
}).assign(removidas_pct=lambda t: (1 - t["depois"] / t["antes"]).round(3)).sort_values("antes", ascending=False)

,antes,depois,removidas_pct
Produto,,,
Café,5378,5161,0.040
Arroz,5305,5053,0.048
Manteiga,4875,4504,0.076
Feijão,4790,4595,0.041
Leite,4651,4463,0.040
Farinha de Trigo,4262,4101,0.038
Óleo,3165,3041,0.039
Açúcar,3101,2978,0.040
Tomate,1332,1287,0.034


## 4. Salvar

A base limpa vai para `data/processed/` e o resumo para `reports/tables/`. As outras etapas leem a base com `data_loading.carregar_base_limpa()`.

In [12]:
config.DIR_DADOS_PROCESSED.mkdir(parents=True, exist_ok=True)
config.DIR_TABELAS.mkdir(parents=True, exist_ok=True)

limpa.to_csv(config.ARQUIVO_BASE_LIMPA, index=False, encoding="utf-8")
resumo.to_csv(config.DIR_TABELAS / "limpeza_resumo.csv", index=False, encoding="utf-8")

# Confere que a base salva volta igual
relida = data_loading.carregar_base_limpa()
print(f"salvo: {config.ARQUIVO_BASE_LIMPA.name} ({len(relida):,} linhas)")

salvo: icb_limpo.csv (41,029 linhas)
